# 🐾 Klasifikasi Hewan: ResNet-50 vs Baseline CNN
**Framework:** TensorFlow/Keras | **Platform:** Google Colab

---
### Kelas yang Diklasifikasi:
- 🐱 Kucing
- 🐶 Anjing
- 🐰 Kelinci
- 🐦 Burung

## 📦 STEP 1: Install & Import Library

In [ ]:
# Import semua library yang dibutuhkan
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import os
import warnings
warnings.filterwarnings('ignore')

print('✅ TensorFlow version:', tf.__version__)
print('✅ GPU:', tf.config.list_physical_devices('GPU'))

## 📁 STEP 2: Upload & Struktur Dataset

Struktur folder yang diharapkan:
```
dataset/
├── train/
│   ├── kucing/
│   ├── anjing/
│   ├── kelinci/
│   └── burung/
├── val/
│   ├── kucing/
│   ├── anjing/
│   ├── kelinci/
│   └── burung/
└── test/
    ├── kucing/
    ├── anjing/
    ├── kelinci/
    └── burung/
```

In [ ]:
# Upload dataset dari komputer lokal (zip)
from google.colab import files
import zipfile

# Uncomment baris di bawah jika upload dari lokal
# uploaded = files.upload()
# with zipfile.ZipFile('dataset.zip', 'r') as zip_ref:
#     zip_ref.extractall('/content/dataset')

# Atau jika sudah ada di Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r '/content/drive/MyDrive/dataset' '/content/dataset'

# ============================================================
# KONFIGURASI PATH - Sesuaikan dengan lokasi dataset kamu
# ============================================================
TRAIN_DIR = '/content/dataset/train'
VAL_DIR   = '/content/dataset/val'
TEST_DIR  = '/content/dataset/test'

# Konfigurasi global
IMG_SIZE    = (224, 224)   # ResNet-50 butuh 224x224
BATCH_SIZE  = 32
EPOCHS      = 30
NUM_CLASSES = 4
CLASS_NAMES = ['anjing', 'burung', 'kelinci', 'kucing']  # sesuaikan nama folder

print('✅ Konfigurasi selesai')

## 🔄 STEP 3: Data Preprocessing & Augmentasi

In [ ]:
# Data Augmentasi untuk training (mencegah overfitting)
train_datagen = ImageDataGenerator(
    rescale=1./255,           # Normalisasi piksel 0-1
    rotation_range=20,        # Rotasi acak
    width_shift_range=0.2,    # Geser horizontal
    height_shift_range=0.2,   # Geser vertikal
    shear_range=0.2,          # Shear
    zoom_range=0.2,           # Zoom acak
    horizontal_flip=True,     # Flip horizontal
    fill_mode='nearest'
)

# Validasi & Test TIDAK diaugmentasi, hanya dinormalisasi
val_test_datagen = ImageDataGenerator(rescale=1./255)

# Load data dari folder
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = val_test_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_generator = val_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False  # Penting! Jangan di-shuffle untuk evaluasi
)

print('✅ Train:', train_generator.samples, 'gambar')
print('✅ Val  :', val_generator.samples, 'gambar')
print('✅ Test :', test_generator.samples, 'gambar')
print('✅ Kelas:', train_generator.class_indices)

## 🖼️ STEP 4: Visualisasi Sample Data

In [ ]:
# Tampilkan beberapa contoh gambar dari dataset
sample_images, sample_labels = next(train_generator)
class_names = list(train_generator.class_indices.keys())

plt.figure(figsize=(12, 6))
for i in range(8):
    plt.subplot(2, 4, i+1)
    plt.imshow(sample_images[i])
    plt.title(class_names[np.argmax(sample_labels[i])], fontsize=12)
    plt.axis('off')
plt.suptitle('Sample Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 🏗️ STEP 5: Bangun Model Baseline CNN

In [ ]:
def build_baseline_cnn(input_shape=(224, 224, 3), num_classes=4):
    model = models.Sequential([
        # Block 1
        layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(2,2),
        layers.Dropout(0.25),

        # Block 2
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(2,2),
        layers.Dropout(0.25),

        # Block 3
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(2,2),
        layers.Dropout(0.25),

        # Block 4
        layers.Conv2D(256, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2,2),
        layers.Dropout(0.25),

        # Classifier Head
        layers.GlobalAveragePooling2D(),
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ], name='Baseline_CNN')

    return model

baseline_model = build_baseline_cnn()
baseline_model.summary()

## 🏗️ STEP 6: Bangun Model ResNet-50 (Transfer Learning)

In [ ]:
def build_resnet50(input_shape=(224, 224, 3), num_classes=4):
    # Load ResNet-50 pretrained ImageNet, tanpa top layer
    base_model = ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )

    # Freeze semua layer base (tidak dilatih ulang)
    base_model.trainable = False

    # Tambahkan custom classifier di atas ResNet-50
    inputs = keras.Input(shape=input_shape)
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs, name='ResNet50_TransferLearning')
    return model, base_model

resnet_model, resnet_base = build_resnet50()
resnet_model.summary()

## ⚙️ STEP 7: Compile Kedua Model

In [ ]:
# Compile Baseline CNN
baseline_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Compile ResNet-50
resnet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print('✅ Kedua model berhasil dikompilasi')

## 🎯 STEP 8: Callbacks

In [ ]:
# Callbacks untuk mencegah overfitting dan menyimpan model terbaik
def get_callbacks(model_name):
    return [
        EarlyStopping(
            monitor='val_accuracy',
            patience=7,
            restore_best_weights=True,
            verbose=1
        ),
        ModelCheckpoint(
            filepath=f'/content/{model_name}_best.h5',
            monitor='val_accuracy',
            save_best_only=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=3,
            min_lr=1e-7,
            verbose=1
        )
    ]

print('✅ Callbacks siap')

## 🚀 STEP 9: Training Baseline CNN

In [ ]:
print('='*50)
print('🚀 Training Baseline CNN...')
print('='*50)

history_baseline = baseline_model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=get_callbacks('baseline_cnn'),
    verbose=1
)

print('\n✅ Training Baseline CNN selesai!')

## 🚀 STEP 10: Training ResNet-50 (Phase 1 - Frozen)

In [ ]:
print('='*50)
print('🚀 Training ResNet-50 Phase 1 (Frozen Base)...')
print('='*50)

history_resnet_phase1 = resnet_model.fit(
    train_generator,
    epochs=15,
    validation_data=val_generator,
    callbacks=get_callbacks('resnet50_phase1'),
    verbose=1
)

print('\n✅ Training Phase 1 selesai!')

## 🔓 STEP 11: Fine-tuning ResNet-50 (Phase 2 - Unfreeze)

In [ ]:
# Unfreeze 30 layer terakhir dari ResNet-50 untuk fine-tuning
resnet_base.trainable = True
for layer in resnet_base.layers[:-30]:
    layer.trainable = False

# Compile ulang dengan learning rate lebih kecil
resnet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print('='*50)
print('🔓 Fine-tuning ResNet-50 Phase 2...')
print('='*50)

history_resnet_phase2 = resnet_model.fit(
    train_generator,
    epochs=15,
    validation_data=val_generator,
    callbacks=get_callbacks('resnet50_phase2'),
    verbose=1
)

print('\n✅ Fine-tuning selesai!')

## 📊 STEP 12: Visualisasi Training History

In [ ]:
def plot_history(history, title):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Akurasi
    ax1.plot(history.history['accuracy'], label='Train Accuracy', color='blue')
    ax1.plot(history.history['val_accuracy'], label='Val Accuracy', color='orange')
    ax1.set_title(f'{title} - Accuracy', fontsize=13, fontweight='bold')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Loss
    ax2.plot(history.history['loss'], label='Train Loss', color='blue')
    ax2.plot(history.history['val_loss'], label='Val Loss', color='orange')
    ax2.set_title(f'{title} - Loss', fontsize=13, fontweight='bold')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'/content/{title.replace(" ","_")}_history.png', dpi=150)
    plt.show()

# Plot masing-masing
plot_history(history_baseline, 'Baseline CNN')
plot_history(history_resnet_phase1, 'ResNet-50 Phase 1')

## 📈 STEP 13: Evaluasi pada Test Set

In [ ]:
# Evaluasi kedua model pada test set
print('='*50)
print('📈 Evaluasi Model pada Test Set')
print('='*50)

baseline_loss, baseline_acc = baseline_model.evaluate(test_generator, verbose=0)
resnet_loss, resnet_acc     = resnet_model.evaluate(test_generator, verbose=0)

print(f'\n🔵 Baseline CNN  → Accuracy: {baseline_acc*100:.2f}% | Loss: {baseline_loss:.4f}')
print(f'🟢 ResNet-50     → Accuracy: {resnet_acc*100:.2f}% | Loss: {resnet_loss:.4f}')

# Tabel perbandingan
results = pd.DataFrame({
    'Model': ['Baseline CNN', 'ResNet-50'],
    'Test Accuracy (%)': [round(baseline_acc*100, 2), round(resnet_acc*100, 2)],
    'Test Loss': [round(baseline_loss, 4), round(resnet_loss, 4)]
})
print('\n', results.to_string(index=False))

## 🔢 STEP 14: Confusion Matrix & Classification Report

In [ ]:
def plot_confusion_matrix(model, generator, model_name, class_names):
    # Prediksi
    generator.reset()
    y_pred = model.predict(generator, verbose=0)
    y_pred_classes = np.argmax(y_pred, axis=1)
    y_true = generator.classes

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred_classes)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix - {model_name}', fontsize=13, fontweight='bold')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.savefig(f'/content/cm_{model_name.replace(" ","_")}.png', dpi=150)
    plt.show()

    # Classification Report
    print(f'\n📋 Classification Report - {model_name}')
    print(classification_report(y_true, y_pred_classes, target_names=class_names))

class_names = list(test_generator.class_indices.keys())

plot_confusion_matrix(baseline_model, test_generator, 'Baseline CNN', class_names)
plot_confusion_matrix(resnet_model, test_generator, 'ResNet-50', class_names)

## 📊 STEP 15: Grafik Perbandingan Akhir

In [ ]:
# Bar Chart perbandingan akurasi
models_name = ['Baseline CNN', 'ResNet-50']
accuracies  = [baseline_acc * 100, resnet_acc * 100]
colors      = ['#4C72B0', '#55A868']

plt.figure(figsize=(8, 6))
bars = plt.bar(models_name, accuracies, color=colors, width=0.5, edgecolor='black')
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{acc:.2f}%', ha='center', va='bottom', fontsize=13, fontweight='bold')
plt.ylim(0, 110)
plt.title('Perbandingan Test Accuracy\nBaseline CNN vs ResNet-50', fontsize=14, fontweight='bold')
plt.ylabel('Accuracy (%)')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('/content/comparison_accuracy.png', dpi=150)
plt.show()

print('\n🏆 Model dengan akurasi lebih tinggi:', models_name[np.argmax(accuracies)])

## 💾 STEP 16: Simpan Model

In [ ]:
# Simpan kedua model
baseline_model.save('/content/baseline_cnn_final.h5')
resnet_model.save('/content/resnet50_final.h5')

print('✅ Model Baseline CNN disimpan: baseline_cnn_final.h5')
print('✅ Model ResNet-50 disimpan   : resnet50_final.h5')

# Download model ke lokal (opsional)
# from google.colab import files
# files.download('/content/baseline_cnn_final.h5')
# files.download('/content/resnet50_final.h5')

## 🔍 STEP 17: Prediksi Gambar Baru (Opsional)

In [ ]:
from tensorflow.keras.preprocessing import image

def predict_image(model, img_path, class_names, model_name='Model'):
    img = image.load_img(img_path, target_size=IMG_SIZE)
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    predictions = model.predict(img_array, verbose=0)[0]
    predicted_class = class_names[np.argmax(predictions)]
    confidence = np.max(predictions) * 100

    plt.figure(figsize=(5, 5))
    plt.imshow(image.load_img(img_path))
    plt.title(f'[{model_name}]\nPrediksi: {predicted_class} ({confidence:.1f}%)',
              fontsize=12, fontweight='bold')
    plt.axis('off')
    plt.show()

    return predicted_class, confidence

# Contoh penggunaan:
# predict_image(resnet_model, '/content/test_image.jpg', class_names, 'ResNet-50')
# predict_image(baseline_model, '/content/test_image.jpg', class_names, 'Baseline CNN')

print('✅ Fungsi prediksi siap digunakan')

---
## ✅ Selesai!
### Ringkasan Pipeline:
1. Import library
2. Load & augmentasi dataset
3. Bangun Baseline CNN dari scratch
4. Bangun ResNet-50 dengan Transfer Learning
5. Training kedua model
6. Fine-tuning ResNet-50
7. Evaluasi & perbandingan hasil
8. Confusion matrix & classification report
9. Simpan model